# IRWA Final Project. Part 2: Indexing and Evaluation

Course: Information Retrieval and Web Analytics (IRWA)
Group members: Sergi Cases, Julia Garcia, Martí Casas

In this part we build an inverted index over the processed corpus from Part 1, implement a conjunctive (AND) search, rank the results with TF-IDF and cosine similarity, and evaluate the search engine with the usual IR metrics.

The code is in the src folder so that it can be reused in Parts 3 and 4:

- src/preprocessing.py: build_terms, copied from Part 1 without changes, used for the queries
- src/indexing.py: loading the corpus, inverted index, idf, document norms and AND search
- src/ranking.py: TF-IDF + cosine ranking
- src/evaluation.py: evaluation metrics

## 0. Setup

In [1]:
import os

import pandas as pd

from src.preprocessing import build_terms
from src.indexing import load_corpus, build_index, compute_idf, compute_doc_norms, conjunctive_search
from src.ranking import rank_tfidf, search
from src.evaluation import (precision_at_k, recall_at_k, avg_precision_at_k, f1_at_k, reciprocal_rank,
                            ndcg_at_k, mean_avg_precision, mean_reciprocal_rank, evaluate_rankings,
                            evaluate_labelled_csv)

DATA_DIR = "./"
CORPUS_PATH = DATA_DIR + "processed_corpus.json"
VALIDATION_PATH = DATA_DIR + "validation_labels.csv"
TEST_RESULTS_PATH = DATA_DIR + "test_queries_results.csv"

pd.set_option("display.max_colwidth", 80)

## 1. Indexing

### 1.1 Load the processed corpus

We load processed_corpus.json (one product per line). The searchable text is the all_terms field, which already contains the processed terms of the title, description, useful product details and brand.

In [2]:
corpus = load_corpus(CORPUS_PATH)
titles = corpus.set_index("pid")["title"]
print("Documents:", len(corpus))
corpus[["pid", "title", "all_terms"]].head(3)

Documents: 28080


,pid,title,all_terms
0,TKPFCZ9EA7H5FYZH,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant, yorker, trackpant, made, rich, comb,..."
1,TKPFCZ9EJZV2UVRZ,Solid Men Blue Track Pants,"[solid, men, blue, track, pant, yorker, trackpant, made, rich, comb, cotton,..."
2,TKPFCZ9EHFCY5Z4Y,Solid Men Multicolor Track Pants,"[solid, men, multicolor, track, pant, yorker, trackpant, made, rich, comb, c..."


### 1.2 Query preprocessing

Queries must be processed exactly like the documents, otherwise a query word would not match the stems stored in the index. We use the same build_terms function of Part 1.

In [3]:
for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue", "Men's Slim Fit T-Shirt"]:
    print(f"{q:38} -> {build_terms(q)}")

women full sleeve sweatshirt cotton    -> ['women', 'full', 'sleev', 'sweatshirt', 'cotton']
men slim jeans blue                    -> ['men', 'slim', 'jean', 'blue']
Men's Slim Fit T-Shirt                 -> ['men', 'slim', 'fit', 'tshirt']


### 1.3 Build the inverted index

For every term we store a posting list with the documents that contain it and the term frequency in each one (term -> {pid: tf}). With the index we also compute:

- idf of every term: log(N / df), where df is the number of documents with the term
- the norm of the tf-idf vector of every document, needed later for the cosine similarity

In [4]:
%%time
index = build_index(corpus, field="all_terms")
idf = compute_idf(index, len(corpus))
doc_norms = compute_doc_norms(index, idf)
print("Vocabulary size:", len(index))

Vocabulary size: 5335
CPU times: total: 531 ms
Wall time: 524 ms


In [5]:
# Example: posting list of a term
term = build_terms("sweatshirt")[0]
print(f"Term '{term}': df = {len(index[term])}, idf = {idf[term]:.3f}")
print("First postings (pid: tf):", list(index[term].items())[:5])

Term 'sweatshirt': df = 1417, idf = 2.987
First postings (pid: tf): [('SWSFYD33P2XCBWFG', 1), ('SWSFYD3KMGTBZEJZ', 1), ('SWSFYD38Y2MAT9FQ', 1), ('SWSFVEV2YFEJKUCK', 2), ('SWSFVEV2BWZJDJ46', 1)]


### 1.4 Conjunctive (AND) search

A document is returned only if it contains all the query terms. The search intersects the posting lists of the query terms, starting with the shortest one. If a term is not in the vocabulary, no document is returned.

In [6]:
for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue", "red wool scarf", "unicornword shirt"]:
    print(f"{q:38} -> {len(conjunctive_search(q, index))} documents")

women full sleeve sweatshirt cotton    -> 549 documents
men slim jeans blue                    -> 220 documents
red wool scarf                         -> 0 documents
unicornword shirt                      -> 0 documents


### 1.5 TF-IDF ranking

The documents returned by the AND search are ranked with the cosine similarity between the tf-idf vector of the query and the tf-idf vector of the document (weight = tf * idf). The document norm uses all its terms, so a product whose text is mostly about the query gets a higher score than a long text that only mentions the query terms. Ties are broken by pid so the order is always the same.

In [7]:
def show_results(query, top_k=10):
    """Prints the top_k ranked results of a query with their titles."""
    ranking = search(query, index, idf, doc_norms, top_k=top_k)
    table = pd.DataFrame(ranking, columns=["pid", "score"])
    table["title"] = table["pid"].map(titles)
    table.index = range(1, len(table) + 1)
    print(f"Query: '{query}' ({len(conjunctive_search(query, index))} documents match all terms)")
    display(table.round(3))


show_results("women full sleeve sweatshirt cotton")
show_results("men slim jeans blue")

Query: 'women full sleeve sweatshirt cotton' (549 documents match all terms)


,pid,score,title
1,SWSFZVTTQCB4SJ7F,0.847,Full Sleeve Solid Women Sweatshirt
2,SWSFZVTNGM4HG8BC,0.832,Full Sleeve Printed Women Sweatshirt
3,SWSFQGS456JAZCQB,0.760,Full Sleeve Printed Women Sweatshirt
4,SWSEXAHDUWHFZDHC,0.759,Full Sleeve Printed Women Sweatshirt
5,SWSEXAHDQQCKCAEA,0.754,Full Sleeve Printed Women Sweatshirt
6,SWSFV5JNFW5HZQWY,0.753,Full Sleeve Printed Women Sweatshirt
7,SWSFMTNHEM5SNHVZ,0.745,Full Sleeve Solid Women Sweatshirt
8,SWSFYRKYAHH4HHSM,0.735,Full Sleeve Printed Women Sweatshirt
9,SWSFYTYMNTBNARUN,0.729,Full Sleeve Solid Women Sweatshirt
10,SWSFY37UKWPG4AZ2,0.728,Full Sleeve Solid Women Sweatshirt


Query: 'men slim jeans blue' (220 documents match all terms)


,pid,score,title
1,JEAFRAQXEKGUPNUN,0.558,Slim Men Blue Jeans
2,JEAFEKQZ4C2Z6GCX,0.544,Slim Men Blue Jeans
3,JEAFEKQZQPYHHVGS,0.544,Slim Men Blue Jeans
4,JEAFS57EYDGY4EJM,0.540,Slim Men Blue Jeans
5,JEAFSKYHSYMTKEJD,0.530,Slim Men Blue Jeans
6,JEAFTGSGTYKZGAEZ,0.530,Slim Men Blue Jeans
7,JEAFM3Y9ZNMVG6ZM,0.528,Slim Men Blue Jeans
8,JEAFSKYHZHSZZC9S,0.525,Slim Men Blue Jeans
9,JEAFJA4M7SMZN2XJ,0.517,Slim Men Black Jeans
10,JEAFKWR3FX9NH2V6,0.516,Slim Men Blue Jeans
